# 27 Oct — Optical-model validation and numerical checks

This notebook holds the verification removed from the two demonstration notebooks. It checks the visible surface-error model, arbitrary two-layer model, finite sampling and Python widget callbacks. Run All independently; no demonstration notebook needs to run first. Results are saved under checks/.

The tests use analytic references, independent algorithms and physical invariants. Passing them validates the declared teaching calculations; it does not establish a model of a real coating with measured optical constants. Python callbacks are tested without claiming browser/frontend automation.


In [1]:
# V01 — Import the models and verification functions.
from pathlib import Path
import sys
BASE_DIR=Path.cwd()
if not (BASE_DIR/'mirror_dent_optics.py').exists():
    candidate=BASE_DIR/'Discussion Series'/'27 Oct'
    if candidate.exists():BASE_DIR=candidate
sys.path.insert(0,str(BASE_DIR))
from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display,Markdown
if get_ipython() is not None:get_ipython().run_line_magic('matplotlib','inline')
import mirror_dent_optics as optics

import json
import verify_optics as verification
REPORT={}


## V02 — Visible surface figure, reflectance and convergence

Check exact Zernike recovery, analytic Gaussian moments, radial integrals, symmetry, piston/tilt behavior, power conservation, masked-pupil QR fitting, convolution centring and independent quadrature/image-grid refinement. Additional checks verify complex-reflectance brightness/global-phase behavior, the pure-defocus global-bow endpoint and integer convergence ticks.

The n≤8 decomposition checks cover all 45 allowed modes, independence from image reconstruction order, and refitting on a masked pupil.


In [2]:
# V02 — Scientific and numerical checks for the visible-light model.
REPORT['visible']=verification.verify_visible()
display(pd.DataFrame([REPORT['visible']]).T)
display(pd.DataFrame(REPORT['visible']['image_grid_refinement']))


,0
gram_max_abs_error,0.0
weight_sum_error,0.0
known_mode_recovery_nm,0.0
gaussian_mean_nm_error,0.0
gaussian_rms_nm_error,0.0
independent_radial_integral_coeff_nm_error,0.0
centered_dent_nonradial_coeff_nm_max,0.0
parseval_residual_variance_nm2_error,0.0
piston_psf_max_abs_error,0.0
positive_tilt_roll_max_abs_error,0.0


,pupil_pixels,on_axis_ratio,absolute_error
0,128,0.776146,0.000226
1,256,0.776236,0.000136
2,384,0.776324,0.000048


## V03 — Two arbitrary films and fixed-total-thickness limits

Compare recursion to independent characteristic matrices, verify passive endpoints and equal-material limits, test exact local total thickness, check the calibrated equal-thickness reflectance maximum, and distinguish a uniform brightness change from a shape change. The default is 40 nm with unnamed, fixed illustrative complex indices.


In [3]:
# V03 — Independent thin-film and image reconstruction checks.
REPORT['coating']=verification.verify_coating()
display(pd.DataFrame([REPORT['coating']]).T)


,0
recursion_vs_matrix_120_cases,0.0
design_total_nm,20.988147
design_wavelength_nm,40.0
design_reflectance,0.007729
equal_split_slope,0.0
sampled_optimum_split,0.0
equal_split_second_derivative,-0.035725
all_A_endpoint_error,0.0
all_B_endpoint_error,0.0
identical_material_split_invariance,0.0


## V04 — Interactive control wiring

Check that visible wavelength, curvature and complex reflectance controls reach the simulation, and that the two-layer slider reaches the all-B endpoint at −100%. Set RENDER_CALLBACKS=True to also exercise plotting through the callbacks. This checks the Python backend, not browser rendering of the frontend controls.

The visible controls must keep all 45 n≤8 coefficients even when the image reconstruction uses a lower order.


In [4]:
# V04 — Widget state/callback checks, separated from the demonstration notebooks.
RENDER_CALLBACKS=False
REPORT['widgets']=verification.verify_widgets(render=RENDER_CALLBACKS)
display(REPORT['widgets'])


{'visible_controls': 'PASS',
 'visible_decomposition_n8_modes': 45,
 'coating_controls': 'PASS',
 'rendered': False,
 'scope': 'Python widget state/callbacks; no frontend browser automation',
 'status': 'PASS'}

## V05 — Import reload regression and three facing mirrors

The saved AttributeError and reproduced cached-module fix are recorded in checks/module_reload_verification.json. The new optical checks verify Fresnel power conservation, a plane-wave reference, the zero-gap OPD-addition limit, all three reflectance factors, the no-dent reference, n≤8 reconstruction of a pure quadratic bow, finite-gap effects, and grid/padding refinement. Widget callbacks exercise all four dent counts. These checks do not claim a full vector or off-axis mirror prescription.


In [5]:
# V05 — Independent physical and numerical checks for the three-reflection panel.
import subprocess
REPORT['module_reload']=json.loads(subprocess.check_output(
    [sys.executable,'-X','utf8',str(BASE_DIR/'verify_module_reload.py')],
    cwd=BASE_DIR,text=True,encoding='utf-8'))
import verify_mirror_train as train_verification
REPORT['three_mirrors']=train_verification.verify_train()
REPORT['three_mirror_widgets']=train_verification.verify_train_widgets()
display(REPORT['three_mirrors'])
display(REPORT['three_mirror_widgets'])


{'fresnel_relative_power_error': 0.0,
 'fresnel_forward_backward_max_error': 2.237726045655905e-15,
 'fresnel_plane_wave_max_error': 3.887366295532235e-15,
 'image_coordinate_vs_single_mirror_error': 0.0,
 'zero_gap_vs_added_opd_max_error': 3.3306690738754696e-16,
 'zero_gap_power_vs_abs_r6_error': 1.1102230246251565e-16,
 'aperture_power_upper_bound': 0.7350918906249998,
 'powers_by_dent_count': {0: 0.7266855877561832,
  1: 0.7266859303311957,
  2: 0.7266860159642,
  3: 0.7266860159642},
 'focal_vs_pupil_power_error': 3.3306690738754696e-16,
 'zero_depth_all_counts_psf_error': 0.0,
 'zero_selected_reconstruction_error': 0.0,
 'common_reflection_phase_intensity_error': 3.885780586188048e-16,
 'three_reflectance_factors_error': 0.0,
 'n8_exact_global_bow_psf_error': 1.1269390654256578e-14,
 'finite_gap_changes_ideal_psf': 0.01936141810535119,
 'grid_refinement': [{'pupil_pixels': 128,
   'padding': 2,
   'on_axis': 0.4072355608384844,
   'throughput': 0.7266860159642},
  {'pupil_pixels'

{'status': 'PASS',
 'dented_counts_tested': [0, 1, 2, 3],
 'shared_depth_nm': 90.0,
 'gap_mm': 250.0,
 'rendered': False,
 'scope': 'Python widget state and callbacks; frontend not automated.'}

## V06 — Save the verification record

The implementation and assertions are in [verify_optics.py](verify_optics.py) and [verify_mirror_dent.py](verify_mirror_dent.py). Dependency versions, exported data and source hashes accompany the result folders. No tests are run inside the two main demonstrations.


In [6]:
# V06 — Save a machine-readable verification record.
CHECK_DIR=BASE_DIR/'checks';CHECK_DIR.mkdir(exist_ok=True)
(CHECK_DIR/'combined_verification.json').write_text(json.dumps(REPORT,indent=2),encoding='utf-8')
print({name:record['status'] for name,record in REPORT.items()})


{'visible': 'PASS', 'coating': 'PASS', 'widgets': 'PASS', 'module_reload': 'PASS', 'three_mirrors': 'PASS', 'three_mirror_widgets': 'PASS'}
